# 03 — Small LoRA training, save, and reload

This is a real **reconstruction-only pilot**, not an experiment result: 16 training
samples, batch size 1, rank 4, and 20 optimizer steps by default. Run it on an
allocated project GPU kernel. Look at the data/baseline notebooks first.

The original photo is the target and its provisional original-color prompt is
the condition. Frozen VAE/text encoders produce a small RAM cache inside this
flow; no separate embedding notebook or full-dataset cache is needed.
Only U-Net attention LoRA is optimized, using diffusion noise-prediction MSE.
Twenty steps checks the plumbing; it does not establish editing quality.

In [ ]:
from pathlib import Path
import json
import os
import sys

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "configs/pilot.json").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter in the project repository, then reopen this notebook.")
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"
os.environ["HF_HOME"] = str(ROOT / "results/_hf_cache")
config = json.loads((ROOT / "configs/pilot.json").read_text())
SHARED = Path(config["shared_root"])
DATA = SHARED / "data/CelebAMask-HQ"
MODEL = SHARED / "models/sd15-inpainting"
if not DATA.is_dir() or not MODEL.is_dir():
    raise FileNotFoundError("Check shared_root in configs/pilot.json.")
from IPython.display import display
from PIL import Image
from color_editing.data import select_samples, describe_sample, contact_sheet
from color_editing.runs import create_run, save_json, finish_run, publish_run
print(json.dumps(config, indent=2))
# To try clothing, set config["region"] = "cloth" here before selecting samples.

In [ ]:
from color_editing.pipeline import train_lora, compare_edits
from color_editing.runs import require_gpu
from collections import Counter

training_samples = select_samples(
    DATA, region=config["region"], split="train", limit=config["training"]["samples"],
    seed=config["split_seed"], min_mask_fraction=config["min_mask_fraction"],
    dilation=config["mask_dilation"],
)
validation_samples = select_samples(
    DATA, region=config["region"], split="val", limit=config["baseline_samples"],
    seed=config["split_seed"], min_mask_fraction=config["min_mask_fraction"],
    dilation=config["mask_dilation"],
)
assert not ({s["image_id"] for s in training_samples} &
            {s["image_id"] for s in validation_samples})
print("Training IDs:", [s["image_id"] for s in training_samples])
print("Provisional labels:", dict(Counter(describe_sample(s)["source_color"] for s in training_samples)))
print("Training settings:", config["training"])
display(contact_sheet([s["image"] for s in training_samples[:4]],
                      [f"train ID {s['image_id']}" for s in training_samples[:4]]))

## Start the 20-step pilot

This cell encodes the selected samples, then starts training. The input is
`noisy latent (4) + mask (1) + masked-image latent (4)`, matching the 9-channel
inpainting U-Net. The training target remains the original photo.

Each step prints loss and gradient norm and appends `loss.jsonl`. Adapter snapshots
are saved every 10 steps and the final adapter goes in `adapter/`. Snapshots hold
adapter weights, not optimizer state; exact training resume is not implemented.
An error or interruption marks the run failed/interrupted and propagates the error.

In [ ]:
device = require_gpu()
TRAIN_RUN = create_run("lora-pilot", config)
print("Saving training run:", TRAIN_RUN)
ADAPTER_DIR = train_lora(training_samples, MODEL, TRAIN_RUN, config)
print("Saved adapter:", ADAPTER_DIR)

In [ ]:
losses = [json.loads(line) for line in (TRAIN_RUN / "loss.jsonl").read_text().splitlines()]
print("step | noise MSE | gradient norm | elapsed seconds")
for row in losses:
    print(f"{row['step']:4d} | {row['loss']:.5f} | {row['grad_norm']:.4f} | {row['elapsed_seconds']:.1f}")
print("Run metadata:")
print((TRAIN_RUN / "run.json").read_text())

## Reload the saved adapter and request a different color

The training function releases its local model references. This cell loads the
base model afresh, generates B1, then reloads the saved adapter for LoRA inference.
It uses the same validation cases and inference settings as notebook 02.

Inspect B0, B1 raw/composite, and LoRA raw/composite. A reconstruction loss alone
does not show that the model follows new target colors; this comparison begins
that check. Synthetic target-color supervision (E1) is a later addition.

In [ ]:
compare_config = {**config, "adapter_source_run": TRAIN_RUN.name,
                  "adapter_directory": str(ADAPTER_DIR)}
COMPARE_RUN = create_run("lora-compare", compare_config)
print("Saving comparison:", COMPARE_RUN)
boards = compare_edits(validation_samples, MODEL, COMPARE_RUN, config, adapter_dir=ADAPTER_DIR)
for path in boards:
    with Image.open(path) as board:
        display(board.copy())
print("Training:", TRAIN_RUN)
print("Comparison:", COMPARE_RUN)

## Optional team copies

Publish the training run to share `adapter/`, and the comparison run to share its
images/records. Teammates can load `shared/runs/<username>/<training-run>/adapter`.
The cells below are commented out by default; nothing is automatically published.

In [ ]:
# shared_training = publish_run(TRAIN_RUN, SHARED)
# shared_comparison = publish_run(COMPARE_RUN, SHARED)
# print(shared_training / "adapter")
# print(shared_comparison)